# 02 — Random LoRA SFT and evaluation

In [ ]:
REPO_URL = "https://github.com/seungjun-green/Korean-TDCS"
!git clone {REPO_URL} korean-math-tdcs
%cd korean-math-tdcs
!pip install -e .

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

drive_results = Path("/content/drive/MyDrive/Korean-TDCS/results")
local_results = Path.cwd() / "results"
drive_results.mkdir(parents=True, exist_ok=True)

if local_results.is_symlink():
    if local_results.resolve() != drive_results.resolve():
        raise RuntimeError(f"{local_results} points to the wrong Drive directory")
elif local_results.exists():
    shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
    shutil.rmtree(local_results)

if not local_results.exists():
    local_results.symlink_to(drive_results, target_is_directory=True)

print(f"Saving all outputs to {drive_results}")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
# ---- User controls ----
TRAINING_BATCH_SIZE = 32  # GPU micro-batch; must be <= effective batch size 32.
TRAINING_MAX_TOKENS = 512  # Total prompt + supervised answer tokens.
EVAL_BATCH_SIZE = 64
EVAL_MAX_TOKENS = 4096  # Maximum newly generated tokens.

if not 1 <= TRAINING_BATCH_SIZE <= 32:
    raise ValueError("TRAINING_BATCH_SIZE must be between 1 and 32")

In [ ]:
audit_cmd = ("python scripts/analyze_difficulty.py --config configs/sft.yaml "
             f"--set training.max_seq_length={TRAINING_MAX_TOKENS}")
!{audit_cmd}

train_cmd = ("python scripts/train_sft.py --config configs/sft.yaml "
             f"--set training.micro_batch_size={TRAINING_BATCH_SIZE} "
             f"--set training.max_seq_length={TRAINING_MAX_TOKENS}")
!{train_cmd}

In [ ]:
BASELINE_RESULTS_PATH = (
    f"results/baseline/max_tokens_{EVAL_MAX_TOKENS}/metrics.json"
)
SFT_RESULTS_PATH = f"results/sft/max_tokens_{EVAL_MAX_TOKENS}/evaluation.json"

cmd = ("python scripts/evaluate.py --config configs/baseline.yaml "
       f"--set evaluation.batch_size={EVAL_BATCH_SIZE} "
       f"--set evaluation.generation.max_new_tokens={EVAL_MAX_TOKENS} "
       "--set model.adapter=results/sft/run_001/adapter "
       f"--set output.results_path={SFT_RESULTS_PATH}")
!{cmd}

In [ ]:
import json
from pathlib import Path

import pandas as pd

base_path = Path(BASELINE_RESULTS_PATH)
base = json.load(base_path.open()) if base_path.exists() else None
sft = json.load(open(SFT_RESULTS_PATH))
table = {'Random SFT': {k: v['score'] for k, v in sft['benchmarks'].items()}}
if base:
    table['Base'] = {k: v['score'] for k, v in base['benchmarks'].items()}
pd.DataFrame(table)